# AI Reliability Judge — Day 1 数据管线 (Kaggle 版)

这个 Notebook 在 Kaggle 上跑完 Day 1 的所有数据生产流程：
1. 从 GitHub 拉最新代码
2. 从 Kaggle Secrets 读 3 个 API key → 生成 `.env`
3. 自检 5 个 provider 是否配置正确
4. MVP 模式跑 60 条（验证管线）
5. 通过后，跑全量 1000 条
6. 下载最终训练数据到 Kaggle Dataset / 本地

---

## 跑之前必须确认

### 右侧面板 → Session options
- ✅ **Internet: ON**（调 API 必须开）
- ❎ Accelerator: None（Day 1 不需要 GPU，省配额给 Day 2 训练）

### 右侧面板 → Add-ons → Secrets（必须有 3 个不同的 Label）

| # | Label | 来源 |
|---|-------|------|
| 1 | `SILICONFLOW_API_KEY` | 硅基流动 cloud.siliconflow.cn |
| 2 | `GOOGLE_API_KEY` | Google AI Studio aistudio.google.com/apikey |
| 3 | `ANTHROPIC_API_KEY` | 中转站 api123.top |

**每个 Secret 右边的开关都必须切到 ON**，不然读不到。

## Cell 1: 自检 Secrets（不泄露真 key）

这一步验证 3 个密钥都配置对了。只显示前 8 位和后 4 位，中间打码。

In [ ]:
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
expected = ["SILICONFLOW_API_KEY", "GOOGLE_API_KEY", "ANTHROPIC_API_KEY"]

all_ok = True
for name in expected:
    try:
        key = s.get_secret(name)
        if not key or len(key) < 10:
            print(f"❌ {name}: value too short ({len(key) if key else 0} chars)")
            all_ok = False
            continue
        masked = key[:8] + "..." + key[-4:]
        print(f"✅ {name}: {masked}  ({len(key)} chars)")
    except Exception as e:
        print(f"❌ {name}: NOT FOUND — {str(e)[:80]}")
        all_ok = False

print()
if all_ok:
    print("✅ All 3 secrets configured. You can continue.")
else:
    print("❌ Fix missing secrets before continuing.")
    print("   Right sidebar → Add-ons → Secrets → check each has a DIFFERENT label")
    print("   and its toggle is ON for this notebook.")

## Cell 2: 拉仓库 + 装依赖

从 GitHub clone 最新的 `feat/hackathon-plan` 分支，安装 pipeline 所需依赖。

Kaggle 容器里已经有大部分科学计算库了，这里只装 pipeline 用的几个轻量包。

In [ ]:
import os

# Kaggle working dir is /kaggle/working (writable + persisted in notebook version)
WORK = "/kaggle/working"
REPO = f"{WORK}/jinan-drone"

# Fresh clone every run (idempotent)
if os.path.exists(REPO):
    !rm -rf {REPO}

!git clone -b feat/hackathon-plan --depth 1 https://github.com/1235789a/jinan-drone.git {REPO}
%cd {REPO}
!git log --oneline -1

# Install only what the pipeline needs (Kaggle already has openai, but version may be old)
!pip install -q --upgrade openai python-dotenv tenacity tqdm

## Cell 3: 从 Secrets 生成 .env 文件

把 Kaggle Secrets 写入 `.env`，这样 pipeline 的 `python-dotenv` 就能读到。

`.env` 在 Kaggle 容器里，只有你自己看得到。Notebook 即使被分享，`.env` 内容也不会被导出（因为它在 working 目录里，但注意：**不要 commit 到 git**，我们的 `.gitignore` 已经排除了）。

In [ ]:
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()

env_content = f"""# Auto-generated from Kaggle Secrets. DO NOT commit.
SILICONFLOW_API_KEY={s.get_secret('SILICONFLOW_API_KEY')}
GOOGLE_API_KEY={s.get_secret('GOOGLE_API_KEY')}
ANTHROPIC_API_KEY={s.get_secret('ANTHROPIC_API_KEY')}
ANTHROPIC_BASE_URL=https://api123.top/v1
ANTHROPIC_MODEL=claude-sonnet-4-6
MAX_CONCURRENT=5
REQUEST_TIMEOUT=60
"""

with open(".env", "w") as f:
    f.write(env_content)

# Verify (show only the public lines)
print(".env created:")
for line in env_content.strip().splitlines():
    if any(line.startswith(prefix) for prefix in ("SILICONFLOW_API_KEY", "GOOGLE_API_KEY", "ANTHROPIC_API_KEY")):
        k, v = line.split("=", 1)
        print(f"  {k}={v[:8]}...{v[-4:]}")
    else:
        print(f"  {line}")

## Cell 4: 验证 5 个 provider 都能加载

跑 `providers.py`，它会打码显示所有 provider 配置。预期看到 **5 个** provider：
- `deepseek` / `glm` / `qwen` 都走 SiliconFlow（同一个 key，不同 model）
- `gemini` 走 Google
- `claude` 走你的中转站

In [ ]:
!python scripts/providers.py

## Cell 5: 跑 MVP —— 60 条数据端到端验证

这一步是**最关键的质量门**。跑 60 条走完整个管线：
1. 生成 60 条种子问题（DeepSeek）
2. 4 数据源并发回答（DeepSeek / GLM / Qwen / Gemini）
3. Claude 做 Meta-Judge 打标签
4. 转成 Gemma chat 训练格式

**耗时约 5-10 分钟**，花费约 ¥1 + $0.5。

⚠️ 这里不用 `!bash run_mvp.sh` 是因为 Kaggle 里 bash 子进程有时读不到 `.env`，我们改成逐步跑。

In [ ]:
# MVP Step 1/4: 生成种子
!python scripts/generate_seeds.py --mvp

In [ ]:
# MVP Step 2/4: 4 数据源并发调用
!python scripts/call_models.py --mvp

In [ ]:
# MVP Step 3/4: Claude Meta-Judge 打标签
!python scripts/label_judge.py --mvp

In [ ]:
# MVP Step 4/4: 格式化为训练数据
!python scripts/format_for_training.py

## Cell 6: MVP 结果验收

合格标准：
- ✅ `total_samples` ≥ 50（允许 15% API 失败）
- ✅ 3 个 risk level 每个占比 15%-55%
- ✅ `balanced == true`

**不合格的话**：查看 `data/stats.json` 的 `domain_x_level` 找出问题 domain，调整种子或 prompt。

In [ ]:
import json

with open("data/stats.json") as f:
    stats = json.load(f)

print("=" * 60)
print("MVP 质量报告")
print("=" * 60)
print(f"Total samples: {stats['total_samples']}")
print(f"Balanced: {stats.get('balanced')}")
print()
print("Risk level distribution:")
for lv in ("low", "medium", "high"):
    pct = stats.get("level_pct", {}).get(lv, 0)
    cnt = stats.get("level_distribution", {}).get(lv, 0)
    bar = "█" * int(pct / 2)
    print(f"  {lv:8s}: {cnt:4d} ({pct:5.1f}%)  {bar}")
print()
print("By domain:")
for d, cnt in stats.get("domain_distribution", {}).items():
    print(f"  {d:10s}: {cnt}")

# Gate: pass or fail
total = stats["total_samples"]
passed = (
    total >= 50
    and stats.get("balanced")
    and all(0.15 <= stats["level_pct"].get(lv, 0) / 100 <= 0.55 for lv in ["low", "medium", "high"])
)
print()
if passed:
    print("✅ MVP PASSED — 可以跑全量 1000 条")
else:
    print("⚠️  MVP marginal — 检查分布后再跑全量")

## Cell 7: （可选）看 3 条真实样本

抽样看一下 Claude 的标签有没有谱。

In [ ]:
import json, random

with open("data/labeled_train.jsonl") as f:
    labeled = [json.loads(l) for l in f if l.strip()]

# Show one sample from each risk level if available
by_level = {"low": [], "medium": [], "high": []}
for x in labeled:
    by_level[x["judgment"]["final_risk_level"]].append(x)

random.seed(7)
for level in ("low", "medium", "high"):
    bucket = by_level[level]
    if not bucket:
        continue
    sample = random.choice(bucket)
    print("=" * 60)
    print(f"[{level.upper()}]  domain={sample['domain']}  lang={sample['lang']}")
    print("Q:", sample["question"])
    for k in ("deepseek", "glm", "qwen", "gemini"):
        v = sample["responses"].get(k, "(missing)")
        print(f"  [{k:8s}] {v[:140]}{'...' if len(v) > 140 else ''}")
    j = sample["judgment"]
    print(f"  -> halluc={j['hallucination_risk']} contra={j['semantic_contradiction']} uncert={j['uncertainty_signals']}")
    print(f"  -> reason: {j.get('reasoning', '')}")
    print()

---

# 🚀 全量模式 —— MVP 通过后再跑

**重要**：看完上面 Cell 6 的 `✅ MVP PASSED` 再跑下面的。

全量 1000 条大约需要：
- 时间：30-60 分钟
- 成本：硅基 ~¥10 + Claude ~$7
- 产出：`data/labeled_train.jsonl` 约 900 行

**断点续传**：如果 Kaggle session 断了，重跑这些 Cell 会从上次断点继续，不会重复调 API。

In [ ]:
# 全量 Step 1/4: 生成 1000 种子
!python scripts/generate_seeds.py

In [ ]:
# 全量 Step 2/4: 4 数据源并发调用 (~15-30 min)
!python scripts/call_models.py

In [ ]:
# 全量 Step 3/4: Claude 打标签 (~10-20 min)
!python scripts/label_judge.py

In [ ]:
# 全量 Step 4/4: 格式化训练数据
!python scripts/format_for_training.py

## Cell 最后: 保存训练数据到 Kaggle Dataset（供 Day 2 训练 notebook 读取）

把 `data/train_chat.jsonl` 和 `data/val_chat.jsonl` 保存到 `/kaggle/working`，Notebook 版本化后 Day 2 的训练 Notebook 可以作为 dataset 引用。

In [ ]:
import shutil, os

os.makedirs("/kaggle/working/reliability_judge_data", exist_ok=True)
for f in ("train_chat.jsonl", "val_chat.jsonl", "labeled_train.jsonl", "stats.json"):
    src = f"data/{f}"
    if os.path.exists(src):
        shutil.copy(src, f"/kaggle/working/reliability_judge_data/{f}")
        size = os.path.getsize(src)
        print(f"  {f}: {size:,} bytes")

print("\nDone. Version this notebook (Save Version) to persist data for Day 2.")